# 🚀 Phase 1: Foundation, Infrastructure & Literature Validation Pipeline
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection (Q1 Publication Pipeline)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

### 📌 Notebook Objectives:
1. **Environment Setup & Google Drive Mounting**: Establish persistent drive storage at `/content/drive/MyDrive/is_ai-vuln/` with NVMe fast symlinks.
2. **Git Safeguards Execution**: Automatically update `.gitignore` so large dataset files (`*.csv`, `*.pcap`, `/data/`) are never committed.
3. **Literature Validation Audit**: Run OpenAlex & CrossRef metadata harvesting and verify active DOIs with 0 retractions for 35 citations.
4. **Data Decontamination & Anti-Leakage Suite**: Verify CICIDS2017 cleaner, Subnet-Grouped K-Fold cross validation, and GraphIDS graph builder.
5. **State Recovery & Publication Styler**: Test `CheckpointManager` and IEEE 300+ DPI layouting engine.

### 1. ☁️ Google Drive Mount & Workspace Setup

In [ ]:
# Mount Google Drive for persistent state & checkpoint recovery
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/is_ai-vuln')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    
    # If running directly inside Drive folder, cd to it
    if Path('/content/is_ai-vuln').exists():
        os.chdir('/content/is_ai-vuln')
    elif DRIVE_ROOT.exists() and (DRIVE_ROOT / 'src').exists():
        os.chdir(str(DRIVE_ROOT))
    print(f'✅ Google Drive mounted. Active directory: {os.getcwd()}')
except ImportError:
    print('ℹ️ Running in local/workstation environment.')

# Ensure repo root is on sys.path
if '.' not in sys.path:
    sys.path.insert(0, '.')
print(f'Python version: {sys.version}')

### 2. 📦 Core Dependencies Installation

In [ ]:
# Install requirements for Phase 1
!pip install -q scikit-learn scipy pandas numpy matplotlib seaborn networkx requests tqdm imbalanced-learn

# Optional packages for subsequent phases
# !pip install -q pyDEMATEL scikit-posthocs optuna xgboost lightgbm

print('✅ Core dependencies installed.')

### 3. 🔒 Git Safeguards & Dataset Cache Initialization

In [ ]:
from src.data.drive_downloader import ensure_gitignore_safeguards, initialize_dataset_directories

# 1. Guarantee that large dataset files will never be staged or pushed
ensure_gitignore_safeguards()

# 2. Initialize raw and processed directories
dirs = initialize_dataset_directories()
print(f'Storage paths ready:\n - Root: {dirs["root"]}\n - Raw: {dirs["raw"]}\n - Processed: {dirs["processed"]}')

### 4. 📚 Literature Citation Audit (35 References)
Harvests metadata and validates active DOI resolution via `doi.org` and checks CrossRef Retraction Watch.

In [ ]:
from src.utils.references_harvester import harvest_and_build_library, TARGET_REFERENCES
from src.utils.references_validator import validate_references

# 1. Build references/library.bib
harvested = harvest_and_build_library(output_bib_file='references/library.bib')

# 2. Validate DOIs and retraction status
report = validate_references(TARGET_REFERENCES, output_report_path='references/validation_report.json')

print(f'\nAudit status: {report["audit_passed"]} (Active DOIs: {report["active_dois"]}/35, Retracted: {report["retracted_count"]})')

### 5. 🧹 Data Decontamination Module (CICIDS2017 Anomaly Fixes)
Resolves zero-length flow duplicates, inf/NaN throughput rates, and zero-variance features.

In [ ]:
import pandas as pd, numpy as np
from src.data.cleaner import decontaminate_cicids2017

# Demonstration on synthetic dirty NetFlow traffic
demo_df = pd.DataFrame({
    ' Source IP ': ['192.168.10.5', '192.168.10.5', '192.168.1.10', '192.168.1.20'],
    ' Flow Duration': [1000, 1000, 2500, 500],
    ' Flow Bytes/s': [50.0, 50.0, np.inf, 120.0],
    'Constant Feature': [0.0, 0.0, 0.0, 0.0],
    ' Label ': ['BENIGN', 'BENIGN', 'PortScan', 'BENIGN']
})

cleaned_df, stats = decontaminate_cicids2017(demo_df, drop_duplicates=True)
print('Cleaning summary statistics:', stats)
display(cleaned_df.head())

### 6. 🛡️ Anti-Leakage Partitioning (`GroupKFold` & Inside-Fold Scaling)
Prevents host session leakage across folds using subnet masks (`/24`) and isolates preprocessing strictly to training folds.

In [ ]:
from src.data.splitters import extract_subnet_mask, AntiLeakageGroupKFold, safe_slice, fit_fold_isolated_pipeline

# 1. Subnet mask extraction
ip_series = pd.Series(['192.168.1.10', '192.168.1.25', '10.0.0.5', '10.0.0.12', '172.16.0.4', '172.16.0.9'])
subnets = extract_subnet_mask(ip_series, mask_prefix_len=24)
print('Extracted Subnet Groups:\n', subnets)

# 2. AntiLeakage GroupKFold verification
X_demo = np.random.randn(6, 4)
y_demo = np.array([0, 1, 0, 1, 0, 1])

gkf = AntiLeakageGroupKFold(n_splits=3)
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_demo, y_demo, groups=subnets)):
    X_tr, y_tr, X_va, y_va, scaler = fit_fold_isolated_pipeline(
        safe_slice(X_demo, tr_idx), safe_slice(y_demo, tr_idx),
        safe_slice(X_demo, val_idx), safe_slice(y_demo, val_idx),
        apply_scaling=True
    )
    print(f'Fold {fold+1}: Train size={len(tr_idx)}, Val size={len(val_idx)}, Scaled Train Mean={X_tr.mean():.4f}')

### 7. 🕸️ Graph Construction for GraphIDS
Converts tabular NetFlow records into directed interaction multigraphs and PyG tensor structures.

In [ ]:
from src.data.graph_builder import build_networkx_flow_graph, export_to_pyg_tensors

flow_df = pd.DataFrame({
    'src_ip': ['192.168.1.10', '192.168.1.20', '192.168.1.10', '10.0.0.5'],
    'dst_ip': ['10.0.0.1', '10.0.0.1', '10.0.0.2', '192.168.1.10'],
    'duration': [100.0, 200.0, 300.0, 150.0],
    'bytes_rate': [5.0, 10.0, 15.0, 8.0],
    'is_attack': [0, 1, 0, 1]
})

G = build_networkx_flow_graph(flow_df, src_ip_col='src_ip', dst_ip_col='dst_ip', label_col='is_attack')
tensors = export_to_pyg_tensors(G)
print(f'Graph Nodes: {tensors["num_nodes"]}, Edge Index Shape: {tensors["edge_index"].shape}, Edge Y: {tensors["edge_y"]}')

### 8. 🔄 State Checkpointing & Autorecovery Pipeline
Verifies state persistence to survive unexpected Google Colab disconnects and preemptions.

In [ ]:
from src.utils.checkpoint_manager import CheckpointManager

chk_dir = Path('./workspace_drive/checkpoints_demo')
mgr = CheckpointManager(chk_dir, dataset_name='DemoDataset', track_name='Track_A', total_folds=5)
mgr.reset_state(backup=False)

# Simulate completing fold 1 and fold 2
mgr.record_fold_completion('TabPFN_v3', 1, {'f1_macro': 0.962, 'latency_ms': 0.45})
mgr.record_fold_completion('TabPFN_v3', 2, {'f1_macro': 0.958, 'latency_ms': 0.43})

print('Should skip fold 1:', mgr.should_skip_fold('TabPFN_v3', 1))
print('Should skip fold 2:', mgr.should_skip_fold('TabPFN_v3', 2))
print('Should skip fold 3 (next to run):', mgr.should_skip_fold('TabPFN_v3', 3))
print('Current recovery state:', json.dumps(mgr.state, indent=2))

# Clean up demo directory
import shutil
if chk_dir.exists():
    shutil.rmtree(chk_dir)

### 9. 📊 Journal-Grade Publication Styler (300+ DPI Vector PDF & PNG)

In [ ]:
import matplotlib.pyplot as plt
from src.visualization.publication_styler import set_publication_style, save_publication_figure

set_publication_style(is_double_column=False)
fig, ax = plt.subplots()
ax.plot([1, 2, 3, 4, 5], [0.91, 0.94, 0.96, 0.97, 0.97], marker='o', label='Mambular SSM ($O(L)$)')
ax.plot([1, 2, 3, 4, 5], [0.88, 0.90, 0.92, 0.93, 0.93], marker='s', label='FT-Transformer ($O(L^2)$)')
ax.set_title('Task-Technology Fit: Adaptation Trajectory')
ax.set_xlabel('Cross-Validation Fold')
ax.set_ylabel('F1 Macro Score')
ax.legend()

save_publication_figure(fig, 'experiment_output/demo_publication_figure')
plt.show()

### 10. 🧪 Comprehensive Phase 1 Test Suite Execution

In [ ]:
# Run full verification suite across all Phase 1 components
!python scratch/test_phase1.py